In [3]:
# pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import openai
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())  # read local .env file

openai.api_key = os.environ["OPENAI_API_KEY"]  # key removed before publishing (2026)
model = 'gpt-3.5-turbo'
openai.api_key = os.environ["OPENAI_API_KEY"]  # key removed before publishing (2026)


In [2]:

# Load the CSV file into a DataFrame
df = pd.read_csv('points-of-interest-google2.csv')

In [4]:
categories_set = {'accounting', 'airport', 'amusement_park', 'aquarium', 'art_gallery', 'atm', 'bakery', 'bank', 'bar', 'beauty_salon', 'bicycle_store', 'book_store', 'bowling_alley', 'bus_station', 'cafe', 'campground', 'car_dealer', 'car_rental', 'car_repair', 'car_wash', 'casino', 'cemetery', 'church', 'city_hall', 'clothing_store', 'convenience_store', 'courthouse', 'dentist', 'department_store', 'doctor', 'drugstore', 'electrician', 'electronics_store', 'embassy', 'fire_station', 'florist', 'funeral_home', 'furniture_store', 'gas_station', 'gym', 'hair_care', 'hardware_store', 'hindu_temple', 'home_goods_store', 'hospital', 'insurance_agency', 'jewelry_store', 'laundry', 'lawyer', 'library', 'light_rail_station', 'liquor_store', 'local_government_office', 'locksmith', 'lodging', 'meal_delivery', 'meal_takeaway', 'mosque', 'movie_rental', 'movie_theater', 'moving_company', 'museum', 'night_club', 'painter', 'park', 'parking', 'pet_store', 'pharmacy', 'physiotherapist', 'plumber', 'police', 'post_office', 'primary_school', 'real_estate_agency', 'restaurant', 'roofing_contractor', 'rv_park', 'school', 'secondary_school', 'shoe_store', 'shopping_mall', 'spa', 'stadium', 'storage', 'store', 'subway_station', 'supermarket', 'synagogue', 'taxi_stand', 'tourist_attraction', 'train_station', 'transit_station', 'travel_agency', 'university', 'veterinary_care', 'zoo'}




In [59]:
print(len(categories_set))

96


In [5]:
businesses_set = set(df['primary_category'])


In [15]:
# converting to a list
businesses_list = list(businesses_set)

# Breaking up the businesses_set so it doesnt exceed token length
businesses1 = set(businesses_list[:500])
businesses2 = set(businesses_list[500:1000])
businesses3 = set(businesses_list[1000:1500])
businesses4 = set(businesses_list[1500:2000])
businesses5 = set(businesses_list[2000:])


In [ ]:
# Get the first 500 elements from primary_category_set
# businesses = set(list(primary_category_set)[:500])

In [10]:
# categories_list = df['primary_category'].tolist()

In [11]:
# categories_set = set(category_list)


In [30]:
# businesses = primary_category_list[:20]

In [52]:
# businesses = primary_category_list[:100]

In [61]:
print(len(businesses_set))

2532


In [7]:
# print(len(primary_category_set))

In [8]:
# primary_category_set = set(primary_category_list)

In [9]:
# businesses_set = set(primary_category_set)

In [11]:
# # Get the first 500 elements from primary_category_set
# businesses = set(list(primary_category_set)[:500])

# # Print the new set
# print(businesses)

In [76]:
# using sets instead of lists

def get_completion(messages, model="gpt-3.5-turbo"):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0.3, # 0 is least creative and 1 is most creative
        max_tokens=1024,
    )
    return response.choices[0].message["content"]


def categorize_businesses(businesses: set, categories: set):
    prompt = "Categorize the following businesses into only the provided categories. If you cant find a category for a business in the list put the business into a category called undefined. The categories are:\n\n"
    
    for category in categories:
        prompt += f"- {category}\n"

    prompt += "\nHere are the businesses to categorize:\n\n"
    for business in businesses:
        prompt += f"- {business}\n"
    
    prompt += "\nPlease provide the categorized businesses in the following format: '<business> - <category>'.\n"

    messages = [
        {"role": "system", "content": "You are an assistant who knows how to listen to instructions."},
        {"role": "user", "content": prompt},
    ]

    response_text = get_completion(messages).strip()

    categorized_businesses = {}
    lines = response_text.split('\n')

    for line in lines:
        if ' - ' in line:
            business, category = line.split(' - ')
            business = business.strip()
            category = category.strip()

            if category not in categorized_businesses:
                categorized_businesses[category] = [business]
            else:
                categorized_businesses[category].append(business)
                
    # Move remaining uncategorized businesses to a separate category 
#     if uncategorized_businesses:
#         categorized_businesses["Uncategorized"] = list(uncategorized_businesses)

    return categorized_businesses


# categorized_businesses = categorize_businesses(businesses_set, categories_set)

# for category, businesses in categorized_businesses.items():
#     print(f"{category}: {businesses}")


In [ ]:
# def get_completion(messages, model="gpt-3.5-turbo"):
#     response = openai.ChatCompletion.create(
#         model=model,
#         messages=messages,
#         temperature=0.3,  # Increased temperature for more creativity
#         max_tokens=1024,
#     )
#     return response.choices[0].message["content"]

# def categorize_businesses(businesses: set, categories: set):
#     prompt_base = "Your task is to categorize a list of businesses into specific categories provided below. It is essential that you only use the given categories and do not create any new ones. If it initially appears that a business does not fit any category, try to think more creatively and find the most suitable category for it from the list. Each business must belong to one of the provided categories, and no new categories should be created. Use only the categories provided below:\n\n"

#     max_iterations = 5
#     iteration = 0
#     uncategorized_businesses = businesses.copy()

#     categorized_businesses = {}

#     while uncategorized_businesses and iteration < max_iterations:
#         prompt = prompt_base
#         for category in categories:
#             prompt += f"- {category}\n"

#         prompt += "\nHere is the list of businesses you need to categorize using only the provided categories:\n\n"
#         for business in uncategorized_businesses:
#             prompt += f"- {business}\n"

#         prompt += "\nWhen categorizing the businesses, please use the following format: '<business> - <category>'.\n"

#         messages = [
#             {"role": "system", "content": "You are a helpful assistant with the ability to follow instructions precisely, without making any modifications."},
#             {"role": "user", "content": prompt},
#         ]

#         response_text = get_completion(messages).strip()

#         lines = response_text.split('\n')

#         uncategorized_businesses_next = set()

#         for line in lines:
#             if ' - ' in line:
#                 business, category = line.split(' - ')
#                 business = business.strip()
#                 category = category.strip()

#                 if category in categories:
#                     if category not in categorized_businesses:
#                         categorized_businesses[category] = [business]
#                     else:
#                         categorized_businesses[category].append(business)
#                     uncategorized_businesses.discard(business)
#                 else:
#                     print(f"Unexpected category '{category}' generated for business '{business}'")
#                     uncategorized_businesses_next.add(business)

#         uncategorized_businesses = uncategorized_businesses_next
#         iteration += 1

# #     # Move remaining uncategorized businesses to a separate category (use this information for manual review)
# #     if uncategorized_businesses:
# #         categorized_businesses["Uncategorized"] = list(uncategorized_businesses)

#     return categorized_businesses


In [77]:
categorized_businesses1 = categorize_businesses(businesses1, categories_set)


In [78]:
# Find the keys in combined_dict that are not in categories_set
unexpected_keys = set(categorized_businesses1.keys()) - categories_set

# Print the unexpected keys
print("Unexpected keys:", unexpected_keys)
print("Number of unexpected keys:", len(unexpected_keys))

# Check if all keys in categories_set are in the combined_dict
missing_keys = categories_set - set(categorized_businesses1.keys())

# Print the missing keys
print("Missing keys:", missing_keys)
print("Number of missing keys:", len(missing_keys))

Unexpected keys: {'undefined'}
Number of unexpected keys: 1
Missing keys: {'drugstore', 'pet_store', 'car_wash', 'atm', 'movie_rental', 'secondary_school', 'shopping_mall', 'bus_station', 'plumber', 'shoe_store', 'electrician', 'florist', 'synagogue', 'bank', 'locksmith', 'moving_company', 'liquor_store', 'beauty_salon', 'convenience_store', 'meal_delivery', 'store', 'home_goods_store', 'laundry', 'funeral_home', 'roofing_contractor', 'school', 'aquarium', 'electronics_store', 'subway_station', 'police', 'mosque', 'casino', 'gas_station', 'embassy', 'fire_station', 'rv_park', 'cafe', 'bicycle_store', 'storage', 'train_station', 'accounting', 'local_government_office', 'bowling_alley', 'campground', 'book_store', 'hospital', 'cemetery', 'airport', 'library', 'light_rail_station', 'city_hall', 'taxi_stand', 'university', 'hardware_store', 'painter', 'department_store', 'pharmacy'}
Number of missing keys: 57


In [73]:
categorized_businesses1

{'real_estate_agency': ['Real estate rental agency', 'Title company'],
 'art_gallery': ['Picture frame shop',
  'Glass blower',
  'Stained glass studio',
  'Painting lessons'],
 'home_goods_store': ['Handyman',
  'Lawn equipment rental service',
  'Lawn mower repair service',
  'Appliance parts supplier',
  'Floor sanding and polishing service',
  'Water purification company',
  'Insulation materials store',
  'Window treatment store',
  'Plumbing supply store',
  'Ventilating equipment manufacturer',
  'Power plant equipment supplier',
  'Lumber store',
  'Sanitation service',
  'Scale supplier',
  'Fire department equipment supplier',
  'Hydraulic equipment supplier',
  'Oriental rug store'],
 'restaurant': ['Steamed bun shop',
  'Korean restaurant',
  'Indian Muslim restaurant',
  'Hawaiian restaurant',
  'Persian restaurant',
  'Hungarian restaurant',
  'African restaurant',
  'Israeli restaurant',
  'Bistro',
  'Lebanese restaurant',
  'Asian restaurant',
  'Taiwanese restaurant',

In [48]:
# Find the keys in combined_dict that are not in categories_set
unexpected_keys = set(categorized_businesses1.keys()) - categories_set

# Print the unexpected keys
print("Unexpected keys:", unexpected_keys)
print("Number of unexpected keys:", len(unexpected_keys))

# Check if all keys in categories_set are in the combined_dict
missing_keys = categories_set - set(categorized_businesses1.keys())

# Print the missing keys
print("Missing keys:", missing_keys)
print("Number of missing keys:", len(missing_keys))

Unexpected keys: set()
Number of unexpected keys: 0
Missing keys: {'drugstore', 'primary_school', 'jewelry_store', 'atm', 'movie_rental', 'secondary_school', 'shopping_mall', 'plumber', 'shoe_store', 'electrician', 'synagogue', 'bank', 'tourist_attraction', 'locksmith', 'moving_company', 'liquor_store', 'meal_delivery', 'laundry', 'funeral_home', 'roofing_contractor', 'aquarium', 'hindu_temple', 'subway_station', 'police', 'mosque', 'casino', 'night_club', 'fire_station', 'rv_park', 'storage', 'accounting', 'bowling_alley', 'campground', 'book_store', 'cemetery', 'convenience_store', 'airport', 'library', 'light_rail_station', 'lawyer', 'taxi_stand', 'university', 'hardware_store', 'painter', 'movie_theater', 'department_store', 'pharmacy'}
Number of missing keys: 47


In [79]:
categorized_businesses2 = categorize_businesses(businesses2, categories_set)


In [80]:
categorized_businesses3 = categorize_businesses(businesses3, categories_set)


In [81]:
categorized_businesses4 = categorize_businesses(businesses4, categories_set)


In [82]:
categorized_businesses5 = categorize_businesses(businesses5, categories_set)


In [83]:
# Combining the dictionaries
combined_dict = {}
for d in [categorized_businesses1, categorized_businesses2, categorized_businesses3, categorized_businesses4, categorized_businesses5]:
    combined_dict.update(d)

# Print the combined dictionary
print(combined_dict)

{'real_estate_agency': ['Real estate developer', 'Real estate auctioneer', 'Mortgage broker'], 'art_gallery': ['Sculpture', 'Stone carving'], 'undefined': ['Portable building manufacturer', 'Cement manufacturer', 'Iron steel contractor', 'Sheet metal contractor', 'Livestock dealer', 'Association', 'Scout hall', 'Web hosting company', 'Contractor', 'Event ticket seller', 'Civil engineer', 'Textile mill', 'Shipping company', 'Barrel supplier', 'Piano repair service', 'Alternative medicine practitioner', 'Food and beverage consultant', 'Food and beverage exporter', 'Party planner', 'Mobility equipment supplier', 'Tunnel', 'Website designer', 'Gun shop', 'Reclamation center', 'Heating contractor', 'Process server', 'Poultry farm', 'Delivery service', 'Fur service', 'Gun club', 'Environmental organization', 'Counselor', 'Nutritionist', 'Recycling collection company', 'Welder', 'Shipping service', 'Arborist and tree surgeon', 'Tool & die shop', 'Metal fabricator', 'Trophy shop', 'Interior fi

In [57]:
# To find the number of keys:
print(len(combined_dict.keys()))

# To find the number of items:
print(len(combined_dict.items()))

83
83


In [58]:
# To find the number of keys:
print(len(categorized_businesses1.keys()))

# To find the number of items:
print(len(categorized_businesses4.items()))

49
45


In [30]:
unique_keys = set()

for d in [categorized_businesses1, categorized_businesses2, categorized_businesses3, categorized_businesses4, categorized_businesses5]:
    unique_keys.update(d.keys())

print("Total unique keys:", len(unique_keys))
print("Unique keys:", unique_keys)


Total unique keys: 271
Unique keys: {'health_food_store', 'medical_clinic', 'technical_school', 'machine_shop', 'atm', 'office_supply_store', 'print_shop', 'gift_shop', 'bakery', 'glass_industry', 'physiotherapist', 'car_parts_store', 'car_rental', 'synagogue', 'music_store', 'hair_care', 'appliance_repair_service', 'sports', 'car_dealer', 'locksmith', 'web_design', 'welding_service', 'consumer_advice_center', 'garden_center', 'employment_agency', 'swimming_pool_contractor', 'store', 'hair_salon', 'nutritionist', 'park', 'restoration_service', 'home_goods_store', 'adult_entertainment_store', 'flour_mill', 'metal_fabricator', 'photography_studio', 'post_office', 'sports_complex', 'professional_services', 'casino', 'chemical_store', 'night_club', 'gas_station', 'legal_services', 'bicycle_store', 'fishing_store', 'excavation_contractor', 'transit_station', 'brewery', 'water_utility_company', 'cosmetics_store', 'animal_rescue_service', 'real_estate_agency', 'city_department_of_public_safet

In [26]:
categorized_businesses4

{'car_repair': ['Auto radiator repair service',
  'Hydraulic repair service',
  'Mechanic'],
 'clothing_store': ['Leather wholesaler',
  's clothing store"',
  'Leather repair service',
  'Fur coat shop'],
 'store': ['Mill', 'Mining company', 'Playground equipment supplier', 'Store'],
 'talent_agency': ['Modeling agency', 'Talent agency'],
 'music_school': ['Music instructor'],
 'sports_store': ['Skateboard shop', 'Golf shop'],
 'land_surveyor': ['Land surveyor'],
 'music_store': ['Record store',
  'DJ supply store',
  'Musical instrument manufacturer',
  'Guitar store'],
 'community_center': ['Veterans center', 'Youth social services organization'],
 'art_gallery': ['Airbrushing service'],
 'spa': ['Massage spa'],
 'office': ['Call center'],
 'food_producer': ['Food producer'],
 'video_production_service': ['Video editing service'],
 'electronics_store': ['Home theater store',
  'Telecommunications equipment supplier'],
 'beauty_salon': ['Wig shop', 'Tanning salon'],
 'lodging': ['Mot

In [75]:
# Find the keys in combined_dict that are not in categories_set
unexpected_keys = set(categorized_businesses1.keys()) - categories_set

# Print the unexpected keys
print("Unexpected keys:", unexpected_keys)
print("Number of unexpected keys:", len(unexpected_keys))

# Check if all keys in categories_set are in the combined_dict
missing_keys = categories_set - set(categorized_businesses1.keys())

# Print the missing keys
print("Missing keys:", missing_keys)
print("Number of missing keys:", len(missing_keys))

Unexpected keys: {'construction', 'business', 'grocery_or_supermarket', 'music_store', 'photography', 'pool', 'theater', 'driving_school', 'counseling', 'bookstore', 'fabric_store', 'animal_shelter', 'chemical_store', 'research'}
Number of unexpected keys: 14
Missing keys: {'drugstore', 'pet_store', 'car_wash', 'meal_takeaway', 'atm', 'movie_rental', 'secondary_school', 'shopping_mall', 'bus_station', 'plumber', 'shoe_store', 'electrician', 'florist', 'synagogue', 'bank', 'locksmith', 'supermarket', 'moving_company', 'liquor_store', 'meal_delivery', 'laundry', 'funeral_home', 'spa', 'roofing_contractor', 'aquarium', 'subway_station', 'police', 'mosque', 'casino', 'embassy', 'fire_station', 'rv_park', 'cafe', 'bicycle_store', 'storage', 'train_station', 'accounting', 'local_government_office', 'bowling_alley', 'campground', 'book_store', 'cemetery', 'beauty_salon', 'airport', 'library', 'light_rail_station', 'convenience_store', 'taxi_stand', 'university', 'painter', 'movie_theater', 'd

In [85]:
# Find the keys in combined_dict that are not in categories_set
unexpected_keys = set(combined_dict.keys()) - categories_set

# Print the unexpected keys
print("Unexpected keys:", unexpected_keys)
print("Number of unexpected keys:", len(unexpected_keys))

# Check if all keys in categories_set are in the combined_dict
missing_keys = categories_set - set(combined_dict.keys())

# Print the missing keys
print("Missing keys:", missing_keys)
print("Number of missing keys:", len(missing_keys))


Unexpected keys: {'garden', 'engineering', 'theater', 'fabric_store', 'sporting_goods_store', 'undefined', 'garbage_dump', 'music_store', 'employment_agency', 'hobby_shop', 'tourist_information_center', 'talent_agency', 'water', 'recycling', 'photography_studio', 'optician', 'butcher_shop', 'art_supply_store', 'manufacturing', 'computer_repair', 'candy_store', 'metal_workshop', 'technology', 'medical_supply', 'child_care', 'office_supply', 'discount_store'}
Number of unexpected keys: 27
Missing keys: {'car_wash', 'train_station', 'aquarium', 'subway_station', 'moving_company', 'mosque', 'light_rail_station', 'library', 'funeral_home', 'painter', 'rv_park', 'plumber', 'pharmacy'}
Number of missing keys: 13


In [84]:
import pickle

# Set the name of the file to save the dictionary as
filename = 'category_dict.pickle'

# Open the file in binary mode and write the dictionary to it using the pickle.dump() function
with open(filename, 'wb') as f:
    pickle.dump(combined_dict, f)


In [86]:
def compare_businesses_set_and_combined_dict(businesses_set, combined_dict):
    missing_in_dict = []
    extra_in_dict = []
    
    # Check if strings in businesses_set can be found in combined_dict
    for business in businesses_set:
        if business not in combined_dict:
            missing_in_dict.append(business)
    
    # Check if there are items in combined_dict not in businesses_set
    for key in combined_dict.keys():
        if key not in businesses_set:
            extra_in_dict.append(key)
    
    return missing_in_dict, extra_in_dict

missing_in_dict, extra_in_dict = compare_businesses_set_and_combined_dict(businesses_set, combined_dict)

if missing_in_dict:
    print(f"Businesses in businesses_set not found in combined_dict: {missing_in_dict}")
else:
    print("All businesses in businesses_set are found in combined_dict")

if extra_in_dict:
    print(f"Businesses in combined_dict not found in businesses_set: {extra_in_dict}")
else:
    print("No extra businesses found in combined_dict")


Businesses in businesses_set not found in combined_dict: [nan, 'Real estate rental agency', 'Picture frame shop', 'Handyman', 'Steamed bun shop', 'Wesleyan church', 'Pediatric dentist', 'Aggregate supplier', 'Cottage', 'Korean restaurant', 'Brewpub', 'Ballroom', 'Auto electrical service', 'Title company', 'Furniture wholesaler', 'Automation company', 'Parking lot', 'Veterinarian', 'Hair care', 'Chemical wholesaler', 'Dessert shop', 'Shipping and mailing service', 'Pool cleaning service', 'Marriage or relationship counselor', 'Lawn equipment rental service', 'Lawn mower repair service', 'Glass blower', 'Occupational medical physician', 'Trucking', 'Dermatologist', 'Brick manufacturer', 'Appliance parts supplier', 'Physical therapy clinic', 'National museum', 'Floor sanding and polishing service', 'Spice store', 'Indian Muslim restaurant', 'Stained glass studio', 'Hawaiian restaurant', 'Recording studio', 'Dental insurance agency', 'Lincoln dealer', 'Driving school', 'Water purification 

In [87]:
combined_dict

{'real_estate_agency': ['Real estate developer',
  'Real estate auctioneer',
  'Mortgage broker'],
 'art_gallery': ['Sculpture', 'Stone carving'],
 'undefined': ['Portable building manufacturer',
  'Cement manufacturer',
  'Iron steel contractor',
  'Sheet metal contractor',
  'Livestock dealer',
  'Association',
  'Scout hall',
  'Web hosting company',
  'Contractor',
  'Event ticket seller',
  'Civil engineer',
  'Textile mill',
  'Shipping company',
  'Barrel supplier',
  'Piano repair service',
  'Alternative medicine practitioner',
  'Food and beverage consultant',
  'Food and beverage exporter',
  'Party planner',
  'Mobility equipment supplier',
  'Tunnel',
  'Website designer',
  'Gun shop',
  'Reclamation center',
  'Heating contractor',
  'Process server',
  'Poultry farm',
  'Delivery service',
  'Fur service',
  'Gun club',
  'Environmental organization',
  'Counselor',
  'Nutritionist',
  'Recycling collection company',
  'Welder',
  'Shipping service',
  'Arborist and tre

In [65]:
categorized_businesses4

{'car_repair': ['Auto radiator repair service',
  'Electric motor repair shop',
  'Boat repair shop',
  'Hydraulic repair service'],
 'real_estate_agency': ['Office space rental agency',
  'Retail space rental agency'],
 'meal_takeaway': ['Juice shop'],
 'clothing_store': ['Leather wholesaler',
  's clothing store"',
  'Leather repair service',
  'Fur coat shop',
  'Clothing wholesaler',
  'Haute couture fashion house',
  'Leather cleaning service'],
 'store': ['Mill',
  'Mining company',
  'Playground equipment supplier',
  'Music instructor',
  'Skateboard shop',
  'Customs broker',
  'Veterans center',
  'Airbrushing service',
  'Call center',
  'Crane service',
  'Video editing service',
  'Non-governmental organization',
  'Biotechnology company',
  'Scrapbooking store',
  'Software company',
  'Building',
  'Photographer',
  'Store',
  'Water testing service',
  'Trade fair construction company',
  'Electric utility manufacturer',
  'Blueprint service',
  'Feed manufacturer',
  '

In [ ]:
# dict1 = {'accounting': ['Tax preparation service'], 'bank': ['Bank'], 'insurance_agency': ['Insurance agency'], 'locksmith': ['Locksmith'], 'roofing_contractor': ['Construction company']}
# dict2 = {'pet_store': ['Equestrian facility', 'Horse boarding stable', 'Fishing store'], 'pharmacy': ['Pharmacy'], 'hospital': ['Medical clinic', 'Medical laboratory'], 'clothing_store': ['Consignment shop']}
# dict3 = {'doctor': ['Doctor'], 'spa': ['Massage spa', 'Spa'], 'lodging': ['Apartment building'], 'physiotherapist': ['Chiropractor'], 'electronics_store': ['Cell phone store'], 'gym': ['Weight loss service'], 'art_gallery': ['Film production company']}

# # Combine the dictionaries
# combined_dict = {}
# for d in [dict1, dict2, dict3]:
#     combined_dict.update(d)

# # Print the combined dictionary
# print(combined_dict)


In [ ]:
# Output
# {'accounting': ['Tax preparation service'], 'bank': ['Bank'], 'insurance_agency': ['Insurance agency'], 'locksmith': ['Locksmith'], 'roofing_contractor': ['Construction company'], 'pet_store': ['Equestrian facility', 'Horse boarding stable', 'Fishing st...}


In [ ]:
# using sets instead of lists

def get_completion(messages, model="gpt-3.5-turbo"):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0.01, # 0 is least creative and 1 is most creative
        max_tokens=1024,
    )
    return response.choices[0].message["content"]

def categorize_businesses(businesses: set, categories: set):
    prompt_base = "Your task is to categorize a list of businesses into specific categories provided below. It is essential that you only use the given categories and do not create any new ones. If it initially appears that a business does not fit any category, try to think more creatively and find the most suitable category for it from the list. Each business must belong to one of the provided categories, and no new categories should be created. Use only the categories provided below:\n\n"

    max_iterations = 5
    iteration = 0
    uncategorized_businesses = businesses.copy()

    categorized_businesses = {}

    while uncategorized_businesses and iteration < max_iterations:
        prompt = prompt_base
        for category in categories:
            prompt += f"- {category}\n"

        prompt += "\nHere is the list of businesses you need to categorize using only the provided categories:\n\n"
        for business in uncategorized_businesses:
            prompt += f"- {business}\n"

        prompt += "\nWhen categorizing the businesses, please use the following format: '<business> - <category>'.\n"

        messages = [
            {"role": "system", "content": "You are a helpful assistant with the ability to follow instructions precisely, without making any modifications."},
            {"role": "user", "content": prompt},
        ]

        response_text = get_completion(messages).strip()

        lines = response_text.split('\n')

        uncategorized_businesses = set()

        for line in lines:
            if ' - ' in line:
                business, category = line.split(' - ')
                business = business.strip()
                category = category.strip()

                if category in categories:
                    if category not in categorized_businesses:
                        categorized_businesses[category] = [business]
                    else:
                        categorized_businesses[category].append(business)
                else:
                    print(f"Unexpected category '{category}' generated for business '{business}'")
                    uncategorized_businesses.add(business)

        iteration += 1

    return categorized_businesses




# def categorize_businesses(businesses: set, categories: set):
#     prompt = "Your task is to categorize a list of businesses into specific categories provided below. It is essential that you only use the given categories and do not create any new ones. If it initially appears that a business does not fit any category, try to think more creatively and find the most suitable category for it from the list. Each business must belong to one of the provided categories, and no new categories should be created. Use only the categories provided below:\n\n"

#     for category in categories:
#             prompt += f"- {category}\n"

#     prompt += "\nHere is the list of businesses you need to categorize using only the provided categories:\n\n"
#     for business in businesses:
#         prompt += f"- {business}\n"

#     prompt += "\nWhen categorizing the businesses, please use the following format: '<business> - <category>'.\n"

#     messages = [
#         {"role": "system", "content": "You are a helpful assistant with the ability to follow instructions precisely, without making any modifications."},
#         {"role": "user", "content": prompt},
#     ]

#     response_text = get_completion(messages).strip()

#     categorized_businesses = {}
#     lines = response_text.split('\n')

#     for line in lines:
#         if ' - ' in line:
#             business, category = line.split(' - ')
#             business = business.strip()
#             category = category.strip()
            
#         if category in categories_set:
#             if category not in categorized_businesses:
#                 categorized_businesses[category] = [business]
#             else:
#                 categorized_businesses[category].append(business)
#         else:
#             print(f"Unexpected category '{category}' generated for business '{business}'")
           
            
            

# #             if category not in categorized_businesses:
# #                 categorized_businesses[category] = [business]
# #             else:
# #                 categorized_businesses[category].append(business)

#     return categorized_businesses


# categorized_businesses = categorize_businesses(businesses_set, categories_set)

# for category, businesses in categorized_businesses.items():
#     print(f"{category}: {businesses}")
